In [2]:
import wrds
import pandas as pd
import numpy as np
import duckdb as dd
import statsmodels.formula.api as smf
db = wrds.Connection()

Loading library list...
Done


In [3]:
# db.list_libraries()
# # db.list_tables('crsp_q_mutualfunds')
# # db.describe_table('crsp_q_mutualfunds','monthly_returns')
# db.list_tables('crsp_a_stock')
# db.describe_table('crsp_a_stock','msf')

# db.list_libraries()
# db.list_tables('wrdssec')


Model Setup and Economic Motivation

We study how BDC market returns relate to NAV while explicitly accounting for the fact that NAV is stale and only updated quarterly. Rather than treating NAV as contemporaneous, we use lagged NAV returns, interpreting NAV as the last observable fundamental signal available to investors at time t. This directly addresses the concern that NAV does not reflect real-time information.

Our approach allows the NAV to price relationship to be state-dependent. Specifically, we examine whether macro conditions—captured by changes in the 10-year Treasury rate (discount rate) and credit spreads (risk premia)—affect how much investors rely on NAV when pricing BDCs.

The interaction terms are central. They test whether the explanatory power of NAV declines when valuation is driven more by discounting and risk premia rather than underlying (and stale) fundamentals.


Does NAVs  explanatory power declines when macro conditions shift investors toward discounting and risk premia instead of stale fundamentals.


![image.png](attachment:image.png)



In [4]:
# df_trade monthly

query = """SELECT
cast((EXTRACT(YEAR FROM date) * 100 + EXTRACT(MONTH FROM date)) as int) AS yyyymm,
    cast(a.date as date) as trade_date,
    a.permno,
    b.ticker,
    ABS(a.prc) AS price,
    a.ret,
    a.vol,
    a.shrout
FROM crsp_a_stock.msf AS a
JOIN crsp_a_stock.dsenames AS b
    ON a.permno = b.permno
    AND a.date BETWEEN b.namedt AND b.nameendt
WHERE b.ticker IN ('ARCC', 'MAIN')
    AND a.date >= '2015-01-01'
ORDER BY b.ticker, a.date
"""
df_trade_monthly = db.raw_sql(query)
df_trade_monthly

,yyyymm,trade_date,permno,ticker,price,ret,vol,shrout
0,201501,2015-01-30,90401,ARCC,16.65,0.066966,499573.0,314108.0
1,201502,2015-02-27,90401,ARCC,17.3,0.039039,386332.0,314108.0
2,201503,2015-03-31,90401,ARCC,17.17,0.017341,452024.0,314469.0
3,201504,2015-04-30,90401,ARCC,17.02,-0.008736,378093.0,314469.0
4,201505,2015-05-29,90401,ARCC,16.75,-0.015864,358205.0,314469.0
...,...,...,...,...,...,...,...,...
235,202408,2024-08-30,92309,MAIN,49.4,-0.029992,96652.0,87074.0
236,202409,2024-09-30,92309,MAIN,50.14,0.026012,69109.0,87074.0
237,202410,2024-10-31,92309,MAIN,51.34,0.028819,73126.0,87074.0
238,202411,2024-11-29,92309,MAIN,55.47,0.085216,88129.0,88178.0


In [5]:
# df_trade daily
query = """
SELECT 
    d.date as trade_date,
    d.permno,
    n.ticker,
    d.prc AS price,
    d.ret,
    d.vol,
    d.shrout
FROM crsp.dsf d
JOIN crsp.dsenames n
    ON d.permno = n.permno
WHERE n.ticker IN ('ARCC','MAIN')
  AND d.date BETWEEN n.namedt AND n.nameendt
  AND d.date >= '2015-12-01'
ORDER BY d.permno, d.date"""

df_trade_daily = db.raw_sql(query)
df_trade_daily


,trade_date,permno,ticker,price,ret,vol,shrout
0,2015-12-01,90401,ARCC,15.87,0.003161,1425885.0,314469.0
1,2015-12-02,90401,ARCC,15.73,-0.008822,1233942.0,314469.0
2,2015-12-03,90401,ARCC,15.68,-0.003179,1548635.0,314469.0
3,2015-12-04,90401,ARCC,15.76,0.005102,1454772.0,314469.0
4,2015-12-07,90401,ARCC,15.36,-0.025381,1968566.0,314469.0
...,...,...,...,...,...,...,...
4567,2024-12-24,92309,MAIN,57.0,0.009922,347649.0,88178.0
4568,2024-12-26,92309,MAIN,57.64,0.011228,439818.0,88178.0
4569,2024-12-27,92309,MAIN,57.9,0.004511,448409.0,88178.0
4570,2024-12-30,92309,MAIN,58.02,0.002073,434029.0,88178.0


In [6]:
# df_rate  (monthly)

query = """
SELECT
    CAST(EXTRACT(YEAR FROM date) * 100 + EXTRACT(MONTH FROM date) AS INT) AS yyyymm,
    date AS rate_date,
    gs10 AS dgs10,
    baa,
    aaa
FROM frb.rates_monthly
WHERE date >= '2015-01-01'
"""

# Use .raw_sql() instead of .raw_query()
df_rate = db.raw_sql(query)

# Calculate spread if needed (baa - aaa)
df_rate['spread'] = df_rate['baa'] - df_rate['aaa']

df_rate.head()

/var/folders/kz/730dlv6x48x44yckc_2pv1n80000gn/T/ipykernel_99397/3546722189.py:18: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_rate['spread'] = df_rate['baa'] - df_rate['aaa']


,yyyymm,rate_date,dgs10,baa,aaa,spread
0,201501,2015-01-31,1.88,4.45,3.46,0.99
1,201502,2015-02-28,1.98,4.51,3.61,0.9
2,201503,2015-03-31,2.04,4.54,3.64,0.9
3,201504,2015-04-30,1.94,4.48,3.52,0.96
4,201505,2015-05-31,2.2,4.89,3.98,0.91


In [7]:
# rate daily

query = """
SELECT 
    CAST(EXTRACT(YEAR FROM date) * 100 + EXTRACT(MONTH FROM date) AS INT) AS yyyymm,
    date AS rate_date,
    dgs10,  
    dbaa, 
    daaa
FROM frb.rates_daily
WHERE cast(date as date) >= '2016-01-01'
ORDER BY date
"""

df_rate_daily = db.raw_sql(query)
df_rate_daily

,yyyymm,rate_date,dgs10,dbaa,daaa
0,201601,2016-01-01,<NA>,<NA>,<NA>
1,201601,2016-01-02,<NA>,<NA>,<NA>
2,201601,2016-01-03,<NA>,<NA>,<NA>
3,201601,2016-01-04,2.24,5.48,4.03
4,201601,2016-01-05,2.25,5.5,4.02
...,...,...,...,...,...
3327,202502,2025-02-09,<NA>,<NA>,<NA>
3328,202502,2025-02-10,4.51,5.96,5.35
3329,202502,2025-02-11,4.54,6.0,5.4
3330,202502,2025-02-12,<NA>,6.07,5.47


In [8]:
# df_nav also calculates nav change


df_nav = pd.read_csv('data/nav.txt')

# Convert date
df_nav['announcement_date'] = pd.to_datetime(df_nav['announcement_date'])

# --------------------------------------------------
# 1) Sort properly (CRITICAL)
# --------------------------------------------------
df_nav = df_nav.sort_values(['ticker', 'announcement_date'])

# --------------------------------------------------
# 2) Compute NAV pct change
# --------------------------------------------------
df_nav['nav_ret'] = df_nav.groupby('ticker')['nav'].pct_change()

# --------------------------------------------------
# 3) Inspect
# --------------------------------------------------
df_nav[['ticker','announcement_date','nav','nav_ret']].head(15)

df_nav.to_csv("data/nav.csv")


/var/folders/kz/730dlv6x48x44yckc_2pv1n80000gn/T/ipykernel_99397/477682467.py:7: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_nav['announcement_date'] = pd.to_datetime(df_nav['announcement_date'])
/var/folders/kz/730dlv6x48x44yckc_2pv1n8

In [9]:
# convert date string to datetime

df_trade_monthly['trade_date'] = pd.to_datetime(df_trade_monthly['trade_date'])
df_trade_daily['trade_date'] = pd.to_datetime(df_trade_daily['trade_date'])
df_nav['announcement_date'] = pd.to_datetime(df_nav['announcement_date'])
df_rate['rate_date'] = pd.to_datetime(df_rate['rate_date'])
df_rate_daily['rate_date'] = pd.to_datetime(df_rate_daily['rate_date'])


/var/folders/kz/730dlv6x48x44yckc_2pv1n80000gn/T/ipykernel_99397/2324378176.py:5: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_nav['announcement_date'] = pd.to_datetime(df_nav['announcement_date'])
/var/folders/kz/730dlv6x48x44yckc_2pv1n

In [10]:
# df_trade_nav_monthly (not needed for now)

query = """
SELECT 
    dt.ticker,
    dt.trade_date,
    dt.price,
    dt.ret,
    dn.nav ,
    dn.announcement_date,
    dn.nav_ret
FROM df_trade_monthly dt
ASOF JOIN df_nav dn 
    ON UPPER(dt.ticker) = UPPER(dn.ticker) 
   AND dt.trade_date >= dn.announcement_date
where extract(year from dt.trade_date) >= 2016
ORDER BY dt.trade_date
"""

df_trade_nav_monthly = dd.query(query)


In [11]:
# df_trade_nav_daily (not needed for now)

query = """
SELECT 
    dt.ticker,
    dt.trade_date,
    dt.price,
    dt.ret,
    dn.nav ,
    dn.announcement_date,
    dn.nav_ret
FROM df_trade_daily dt
ASOF JOIN df_nav dn 
    ON UPPER(dt.ticker) = UPPER(dn.ticker) 
   AND dt.trade_date >= dn.announcement_date
where extract(year from dt.trade_date) >= 2016
ORDER BY dt.trade_date
"""

df_trade_nav_daily = dd.query(query).to_df()

df_trade_nav_daily

,ticker,trade_date,price,ret,nav,announcement_date,nav_ret
0,MAIN,2016-01-04,29.18,0.003439,21.56,2015-11-05,0.002325
1,ARCC,2016-01-04,14.46,0.014737,16.79,2015-11-04,-0.000595
2,MAIN,2016-01-05,29.67,0.016792,21.56,2015-11-05,0.002325
3,ARCC,2016-01-05,14.59,0.008990,16.79,2015-11-04,-0.000595
4,MAIN,2016-01-06,29.77,0.003370,21.56,2015-11-05,0.002325
...,...,...,...,...,...,...,...
4523,ARCC,2024-12-27,22.02,0.003646,19.74,2024-10-30,0.006629
4524,ARCC,2024-12-30,21.94,-0.003633,19.74,2024-10-30,0.006629
4525,MAIN,2024-12-30,58.02,0.002073,29.80,2024-11-07,0.000000
4526,ARCC,2024-12-31,21.89,-0.002279,19.74,2024-10-30,0.006629


In [12]:
# pre-event study dataset
#nav event + trade daily ###### MAIN DF ######### 

#[** 2nd way is to to use the nav from previous for pre-event]

df_event_study = dd.query("""
    SELECT *
    FROM df_nav
    INNER JOIN df_trade_daily
        ON df_nav.ticker = df_trade_daily.ticker
       AND df_trade_daily.trade_date >= df_nav.announcement_date - INTERVAL '3 day'
       AND df_trade_daily.trade_date <= df_nav.announcement_date + INTERVAL '3 day'
""").to_df()

df_event_study


,ticker,quarter_end_date,nav,announcement_date,nav_ret,trade_date,permno,ticker_1,price,ret,vol,shrout
0,ARCC,2023-09-30,18.99,2023-10-24,0.022067,2023-10-23,90401,ARCC,18.81,0.001064,5049117.0,569000.0
1,ARCC,2023-09-30,18.99,2023-10-24,0.022067,2023-10-24,90401,ARCC,18.99,0.009569,4099140.0,569437.0
2,ARCC,2023-09-30,18.99,2023-10-24,0.022067,2023-10-25,90401,ARCC,18.90,-0.004739,2810351.0,569437.0
3,ARCC,2023-09-30,18.99,2023-10-24,0.022067,2023-10-26,90401,ARCC,18.89,-0.000529,2285730.0,569437.0
4,ARCC,2023-09-30,18.99,2023-10-24,0.022067,2023-10-27,90401,ARCC,18.66,-0.012176,3004359.0,569437.0
...,...,...,...,...,...,...,...,...,...,...,...,...
348,MAIN,2023-03-31,27.23,2023-05-04,0.013775,2023-05-05,92309,MAIN,40.33,0.021022,406613.0,79550.0
349,MAIN,2022-12-31,26.86,2023-02-23,0.035466,2023-02-21,92309,MAIN,39.36,-0.022840,354156.0,77254.0
350,MAIN,2022-12-31,26.86,2023-02-23,0.035466,2023-02-22,92309,MAIN,39.81,0.011433,243010.0,77254.0
351,MAIN,2022-12-31,26.86,2023-02-23,0.035466,2023-02-23,92309,MAIN,40.19,0.009545,306558.0,77254.0


We analysize df_trade_nav

In [13]:
# trade daily + nav event EVENT STUDY

# 1. Select only necessary columns
# Note: 'ticker' and 'ticker_1' are duplicates from the join; we keep one.

cols_to_keep = [
    'ticker', 
    'announcement_date', 
    'trade_date', 
    'nav', 
    'nav_ret', 
    'ret', 
    'price'
]

df_event_study = df_event_study[cols_to_keep].copy()

# 2. Recalculate days_from_event for consistent indexing
df_event_study['days_from_event'] = (pd.to_datetime(df_event_study['trade_date']) - 
                                     pd.to_datetime(df_event_study['announcement_date'])).dt.days

# 3. Sort for logical flow
df_event_study = df_event_study.sort_values(['ticker', 'announcement_date', 'trade_date'])

# 4. Calculate Cumulative Return starting from the beginning of the [-3, +3] window
df_event_study['cum_ret'] = df_event_study.groupby(['ticker', 'announcement_date'])['ret'].transform(lambda x: (1 + x).cumprod() - 1)

df_event_study.head(7)

/var/folders/kz/730dlv6x48x44yckc_2pv1n80000gn/T/ipykernel_99397/1171897969.py:19: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df_event_study['days_from_event'] = (pd.to_datetime(df_event_study['trade_date']) -
/var/folders/kz/730dlv6x48x4

,ticker,announcement_date,trade_date,nav,nav_ret,ret,price,days_from_event,cum_ret
154,ARCC,2016-02-24,2016-02-22,16.46,-0.019655,0.006275,12.83,-2,0.006275
155,ARCC,2016-02-24,2016-02-23,16.46,-0.019655,0.007015,12.92,-1,0.013334
156,ARCC,2016-02-24,2016-02-24,16.46,-0.019655,0.000000,12.92,0,0.013334
157,ARCC,2016-02-24,2016-02-25,16.46,-0.019655,0.036378,13.39,1,0.050197
158,ARCC,2016-02-24,2016-02-26,16.46,-0.019655,0.008962,13.51,2,0.059609
149,ARCC,2016-05-04,2016-05-02,16.50,0.002430,-0.000658,15.18,-2,-0.000658
150,ARCC,2016-05-04,2016-05-03,16.50,0.002430,-0.006588,15.08,-1,-0.007242


Interpretation:

NAV is stale and it does not explain return which is what we expected.
    We could include only 

In [14]:
# only event records since day -1.

df_reg_daily_event = dd.query("""select 
* from df_event_study
where days_from_event >= -1
order by ticker, announcement_date, days_from_event
""").to_df().head(10)

In [15]:
import statsmodels.api as sm
import pandas as pd
import numpy as np

# 1. Ensure the dataset is clean and dummies are integers
# Using drop_first=False because we manually select days, and T-1 is our excluded baseline
df_reg = pd.get_dummies(df_event_study, columns=['days_from_event'], prefix='day')

# 2. Define features (T=0, T=1, T=2)
X_cols = ['day_0', 'day_1', 'day_2']
X = df_reg[X_cols].astype(float) # Force to float to fix the ValueError
X = sm.add_constant(X)

# 3. Define target and force to float
y = pd.to_numeric(df_reg['ret'], errors='coerce').astype(float)

# 4. Handle any potential NaNs created by coerce or existing in data
# 'missing=drop' in OLS handles this, but statsmodels prefers clean arrays
valid_idx = y.notna()
y = y[valid_idx]
X = X[valid_idx]

# 5. Run the OLS
model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    ret   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     1.395
Date:                Sun, 12 Apr 2026   Prob (F-statistic):              0.244
Time:                        21:52:39   Log-Likelihood:                 942.88
No. Observations:                 353   AIC:                            -1878.
Df Residuals:                     349   BIC:                            -1862.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0007      0.001      0.532      0.5

interpretation:
Day -1 return :- Average return on the day before announcement is negligible .07% (Not signnificant)
Day 0 return :- Average return on the day of announcement is .22% (Not significant)
Day 1 return :- Average return jumps by .46% or 46 basis points.
Mean revision :- 0.0%

Is Day 1 return .46% a NAV surprise?
Regress market return (t+1) = nav_ret

Is it the nav_change that caused the jump on return by the .46% i.e. 46 basis points.

In [16]:
# 1.regress return on day 1 nav_ret

df_day1 = df_event_study[df_event_study['days_from_event'] == 1].copy()

# 2. Define features: The NAV Surprise
X = df_day1['nav_ret'].astype(float)
X = sm.add_constant(X)

# 3. Define target: The Day 1 Return
y = df_day1['ret'].astype(float)

# 4. Run the OLS
model_sensitivity = sm.OLS(y, X, missing='drop').fit()

print(model_sensitivity.summary())

                            OLS Regression Results                            
Dep. Variable:                    ret   R-squared:                       0.036
Model:                            OLS   Adj. R-squared:                  0.022
Method:                 Least Squares   F-statistic:                     2.585
Date:                Sun, 12 Apr 2026   Prob (F-statistic):              0.112
Time:                        21:52:39   Log-Likelihood:                 184.37
No. Observations:                  72   AIC:                            -364.7
Df Residuals:                      70   BIC:                            -360.2
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0063      0.002      2.704      0.0

Interpretation:
    
    1. alpha on day 1 is .63% => book value is does not matter.
    2. NAV is -.14%
    
    We conclude that NAV has no bearing on the price.


In [17]:
# data integrity #

# 1. Check for Look-Ahead Bias (Did we match with a future NAV?)
future_matches = df_reg[df_reg['trade_date'] < df_reg['announcement_date']]

# 2. Check for missing months per ticker since 2016
# Expecting 108 months per ticker (9 years * 12 months)
obs_count = df_reg.groupby('ticker')['trade_date'].count()

# 3. Check for Nulls in key regression columns
nulls = df_reg[['price', 'nav', 'ret', 'nav_ret']].isnull().sum()

print("--- DATA INTEGRITY TEST RESULTS ---")
print(f"Look-ahead bias records: {len(future_matches)}")
print("\nObservations per Ticker (Expected 108):")
print(obs_count)
print("\nMissing Values in Regression Columns:")
print(nulls)

if len(future_matches) == 0 and (obs_count == 108).all():
    print("\n✅ SUCCESS: No records dropped and date logic is sound.")
else:
    print("\n⚠️ WARNING: Check for missing months or join errors.")

--- DATA INTEGRITY TEST RESULTS ---
Look-ahead bias records: 156

Observations per Ticker (Expected 108):
ticker
ARCC    179
MAIN    174
Name: trade_date, dtype: int64

Missing Values in Regression Columns:
price      0
nav        0
ret        0
nav_ret    0
dtype: int64

⚠️ WARNING: Check for missing months or join errors.




In this sectionm we add 
1. federal reserve 10 year tbill rate
2. credit spread 


In [18]:
# trade + rate (daily)

query = """
SELECT 
    t.trade_date,
    t.ret,
    t.announcement_date,
    d.dgs10,
    d.dbaa,
    -- Added missing comma above and fixed the parentheses/brackets below
    CAST(d.dbaa AS DECIMAL(10,3)) - CAST(d.dgs10 AS DECIMAL(10,3)) AS spread
FROM df_trade_nav_daily t
JOIN df_rate_daily d ON t.trade_date = d.rate_date
"""

df_combined = dd.query(query).to_df()
df_combined

,trade_date,ret,announcement_date,dgs10,dbaa,spread
0,2016-01-04,0.003439,2015-11-05,2.24,5.48,3.24
1,2016-01-04,0.014737,2015-11-04,2.24,5.48,3.24
2,2016-01-05,0.016792,2015-11-05,2.25,5.50,3.25
3,2016-01-05,0.008990,2015-11-04,2.25,5.50,3.25
4,2016-01-06,0.003370,2015-11-05,2.18,5.44,3.26
...,...,...,...,...,...,...
4523,2024-12-27,0.003646,2024-10-30,4.62,6.02,1.40
4524,2024-12-30,-0.003633,2024-10-30,4.55,5.98,1.43
4525,2024-12-30,0.002073,2024-11-07,4.55,5.98,1.43
4526,2024-12-31,-0.002279,2024-10-30,4.58,6.00,1.42
